# 07 机器学习势（MLFF）入门：CHGNet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/07_mlff_chgnet.ipynb)

## Learning objectives
理解 MLFF 学习的是势能面近似，区分 energy/force/stress 标签，并建立“预训练模型必须在目标体系验证”的意识。

## 1. 从 DFT 到 ML potential
DFT 对构型 $R$ 给出能量 $E(R)$；力满足 $F_i=-\partial E/\partial R_i$。MLFF 用大量 DFT-labelled structures 学习这个映射，从而以远低于 DFT 的成本反复计算 energy/forces，用于 relaxation 或 MD。

因此 MLFF 的关键不是单个结构预测，而是 **training domain 是否覆盖 MD 会访问的构型空间**。

In [ ]:
!pip -q install chgnet pymatgen

In [ ]:
from pymatgen.core import Lattice, Structure
from chgnet.model import CHGNet
# 小型无机结构仅用于演示 API，不代表 COF benchmark。
structure=Structure(Lattice.cubic(4.2),['Li','Cl'],[[0,0,0],[0.5,0.5,0.5]])
model=CHGNet.load()
pred=model.predict_structure(structure)
print('Energy/atom:',pred['e'])
print('Forces shape:',pred['f'].shape)

## 2. 对 COF 使用预训练 MLFF 前的 checklist
1. 元素和化学环境是否在训练域内？
2. 是否覆盖层间 vdW、滑移和柔性形变？
3. guest/water/ions 是否覆盖？
4. 若涉及成键变化/质子转移，训练数据是否包含这些构型？
5. 在你自己的 DFT test set 上 energy/force error 是多少？
6. MD 中是否出现明显 extrapolation / 非物理结构？
7. 是否需要 fine-tuning / active learning？

**API 成功返回数字不是验证。**

## 3. 研究级 MLFF workflow
`DFT seed data → train/fine-tune → independent test → exploratory MD → uncertainty/extrapolation detection → add DFT labels → retrain → production MD`

对新生，本节只要求理解 workflow。训练 DeepMD/MACE/CHGNet fine-tuning 应放到掌握 DFT 和 MD 之后。

## Exercises
1. 解释 energy MAE 与 force MAE 各自影响什么。
2. 为什么只测试 equilibrium structures 不足以验证 MD 势？
3. 为“hydrated COF + CO₂”设计一个最小 DFT validation set。
4. 比较 pretrained universal MLFF、fine-tuning、从头训练三条路线。

### Take-home message
MLFF 是势能面的数据驱动近似。它的可信度由训练覆盖与独立验证决定，而不是模型名称决定。